# MISC

In [20]:
import os, sys
import pandas as pd
import numpy as np
import json
import shutil
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed

notebook_dir = os.getcwd()  # Gets test folder path
project_dir = os.path.dirname(notebook_dir)  # Gets Project folder path
sys.path.append(project_dir)

from subproblems import *
from utils import *

In [2]:
# constraint based post-proc function
def post_process_constraint(pred, param, pros_decision = None, sce_idx = None, p2p_bal = True):
    h = int(param['hour'])
    P2PTrade = param['P2PTrade']
    P_load = param['load_demamd']
    PI_buy = param['buy_priority']
    PI_sell = param['sell_priority']
    ub_CESc, lb_CESc = param['ub_CESc'], param['lb_CESc']
    ub_CESd, lb_CESd = param['ub_CESd'], param['lb_CESd']

    pred = pred.reshape(-1, 192, 32) 

    qc, qd, P_buy, P_sell = [pred[:, i*h:(i+1)*h, :] for i in range(4)]
    if pros_decision is not None: Pg_buy, Pg_sell  = [pros_decision[i*h:(i+1)*h, :] for i in range(2)]

    # Post-process for boundary
    qc = np.clip(qc, a_min=lb_CESc[0], a_max=ub_CESc[0])
    qd = np.clip(qd, a_min=lb_CESd[0], a_max=ub_CESd[0])
    P_buy = np.clip(P_buy, a_min=0, a_max=None)
    P_sell = np.clip(P_sell, a_min=0, a_max=None)
    # print(qc.shape, qd.shape, P_buy.shape, P_sell.shape)

    # Post-process for P2P trade hour
    start, end_ = P2PTrade 
    for t in range(0, start): # 0 ~ 15 -> 1 ~ 16
        P_buy[:,t,:] = 0
        P_sell[:,t,:] = 0
    for t in range(end_ - 1, h):  # 37 ~ 47 -> 38 ~ 48
        P_buy[:,t,:] = 0
        P_sell[:,t,:] = 0

    if p2p_bal == True:
        # Post-process for priority based P2P balancing
        for sce in range(pred.shape[0]):
            for t in range(h):
                p2p_balance = P_buy[sce,t,:].sum() - P_sell[sce,t,:].sum()
                # print(f"Scenario {sce}, Hour {t}: P2P Balance = {p2p_balance}")
                bp_idx_sorted = np.argsort(PI_buy[t,:])  # Ascending order as highest priority has the lowest value
                sp_idx_sorted = np.argsort(PI_sell[t,:])[::-1]  # Descending order as highest priority has the highest value
                    
                if p2p_balance < 0: # more selling than buying, need to increase buying or decrease selling
                    # Adjust P_sell to balance P2P trade
                    for idx in sp_idx_sorted:
                        if p2p_balance >= 0:
                            break
                        reduction = min(P_sell[sce, t, idx], -p2p_balance)  # Reduce as much as needed or as much as possible
                        P_sell[sce, t, idx] -= reduction
                        p2p_balance += reduction
                    
                    # If still imbalance, adjust P_buy to balance P2P trade
                    for idx in bp_idx_sorted:
                        if p2p_balance >= 0:
                            break
                        increase = min(ub_CESd - qd[sce, t, idx], -p2p_balance)  # Increase as much as needed or as much as possible within bounds
                        P_buy[sce, t, idx] += increase
                        p2p_balance += increase

                if p2p_balance > 0: # more buying than selling, need to increase selling or decrease buying
                    # Adjust P_buy to balance P2P trade
                    for idx in sp_idx_sorted:
                        if p2p_balance <= 0:
                            break
                        reduction = min(P_buy[sce, t, idx], p2p_balance)  # Reduce as much as needed or as much as possible
                        P_buy[sce, t, idx] -= reduction
                        p2p_balance -= reduction

                    # If still imbalance, adjust P_sell to balance P2P trade
                    for idx in bp_idx_sorted:
                        if p2p_balance <= 0:
                            break
                        increase = min(ub_CESc - qc[sce, t, idx], p2p_balance)  # Increase as much as needed or as much as possible within bounds
                        P_sell[sce, t, idx] += increase
                        p2p_balance -= increase

    return np.concatenate([qc, qd, P_buy, P_sell], axis=1)
                

In [3]:
# constraint feasibility check function
def check_constraint_feasibility_np(preds, external_data_np, tolerance=1e-3, ver='Original'):
    """
    Checks if provided predictions satisfy physical constraints using NumPy.
    
    Args:
        preds: NumPy array of denormalized predictions shape (Batch, 6144)
        indices: NumPy array or list of indices 
        external_data_np: NumPy array shape (Total_Scenarios, 48, 32)
        tolerance: Acceptable error margin
    """
    # 1. Ensure shape is (Batch, 192, 32)
    # 192 features: [Charge(48), Discharge(48), Buy(48), Sell(48)]
    # We use .reshape to handle both (Batch, 1, 6144) and (Batch, 6144)
    pred_matrix = preds.reshape(-1, 192, 32)
    
    # 2. Extract Variables (Batch, 48, 32)
    p_chg  = pred_matrix[:, 0:48, :]
    p_dis  = pred_matrix[:, 48:96, :]
    p_buy  = pred_matrix[:, 96:144, :]
    p_sell = pred_matrix[:, 144:192, :]
    
    # --- CHECK 1: Market Balance (Total Buy = Total Sell) ---
    total_buy = np.sum(p_buy, axis=2)   # Sum across 32 users -> (Batch, 48)
    total_sell = np.sum(p_sell, axis=2) # Sum across 32 users -> (Batch, 48)
    
    diff_balance = np.abs(total_buy - total_sell)
    max_bal_err = np.max(diff_balance)
    avg_bal_err = np.mean(diff_balance)
    
    # --- CHECK 2: Net Load Clearing (Supply == Demand) ---
    # Eq: (Buy - Sell) + (Discharge - Charge) = NetLoad
    energy_supplied = (p_buy - p_sell) + (p_dis - p_chg)
    
    # Fetch specific external data for this batch
    batch_net_load = external_data_np # (Batch, 48, 32)
    
    diff_clearing = np.abs(energy_supplied - batch_net_load)
    max_clr_err = np.max(diff_clearing)
    avg_clr_err = np.mean(diff_clearing)

    # --- Summary Report ---
    print("-" * 50)
    print(f"PHYSICAL CONSTRAINT REPORT ({ver})")
    print("-" * 50)
    
    passed_bal = max_bal_err < tolerance
    print(f"1. Market Balance: {'✅ PASS' if passed_bal else '❌ FAIL'}")
    print(f"   Max Error: {max_bal_err:.6f} | Avg Error: {avg_bal_err:.6f}")
    
    passed_clr = max_clr_err < tolerance
    print(f"\n2. Net Load Clearing: {'✅ PASS' if passed_clr else '❌ FAIL'}")
    print(f"   Max Error: {max_clr_err:.6f} | Avg Error: {avg_clr_err:.6f}")
    print("-" * 50)
    
    return {
        "balance_passed": passed_bal,
        "clearing_passed": passed_clr,
        "max_balance_error": max_bal_err,
        "max_clearing_error": max_clr_err
    }

In [30]:
# optimal_iter folder summarize
def gather_results(datapath, start_sce_save, end_sce_save):
    all_de, all_dr, all_pe, all_pr, all_obj = [], [], [], [], []
    i = start_sce_save
    while i <= end_sce_save - 4:
        first = i
        last = i + 4
        
        path_de = f"{datapath}/location/dual_error_{first}to{last}sce.csv"
        path_dr = f"{datapath}/location/dual_residual_{first}to{last}sce.csv"
        path_pe = f"{datapath}/location/primal_error_{first}to{last}sce.csv"
        path_pr = f"{datapath}/location/primal_residual_{first}to{last}sce.csv"
        path_obj = f"{datapath}/location/objective_value_{first}to{last}sce.csv"
        
        # Read CSVs
        df_de = pd.read_csv(path_de).values
        df_dr = pd.read_csv(path_dr).values
        df_pe = pd.read_csv(path_pe).values
        df_pr = pd.read_csv(path_pr).values
        df_obj = pd.read_csv(path_obj).values

        all_de = np.concatenate((all_de, df_de), axis=0) if len(all_de) > 0 else df_de
        all_dr = np.concatenate((all_dr, df_dr), axis=0) if len(all_dr) > 0 else df_dr
        all_pe = np.concatenate((all_pe, df_pe), axis=0) if len(all_pe) > 0 else df_pe
        all_pr = np.concatenate((all_pr, df_pr), axis=0) if len(all_pr) > 0 else df_pr
        all_obj = np.concatenate((all_obj, df_obj), axis=0) if len(all_obj) > 0 else df_obj
        
        i += 5

        df_all_de = pd.DataFrame(all_de)
        df_all_dr = pd.DataFrame(all_dr)
        df_all_pe = pd.DataFrame(all_pe)
        df_all_pr = pd.DataFrame(all_pr)
        df_all_obj = pd.DataFrame(all_obj)

        df_all_de.to_csv(f"{datapath}/optimal_iter/dual_error_{start_sce_save}to{end_sce_save}sce.csv", index=False)
        df_all_dr.to_csv(f"{datapath}/optimal_iter/dual_residual_{start_sce_save}to{end_sce_save}sce.csv", index=False)
        df_all_pe.to_csv(f"{datapath}/optimal_iter/primal_error_{start_sce_save}to{end_sce_save}sce.csv", index=False)
        df_all_pr.to_csv(f"{datapath}/optimal_iter/primal_residual_{start_sce_save}to{end_sce_save}sce.csv", index=False)
        df_all_obj.to_csv(f"{datapath}/optimal_iter/objective_value_{start_sce_save}to{end_sce_save}sce.csv", index=False)
        

# Data Loading

In [4]:
datapath = r"D:\Jacky\Data Output\ADMM_P2P\Database\LP_PrioGO_test_20_OldSame"

with open(f'{datapath}/config.json', 'r') as file:
    config = json.load(file)

SCE_START, SCE_END = config["sce_start"], config["sce_end"]
BUS_SYS = 33
MAX_ITER = 5000
PRIMAL_TOL, DUAL_TOL = 1e-3, 1e-3
BATTERY_CAP = 2
P2P_TRADE = (16, 38)
SOLAR_SCALER = 1.5
DATA_DIR = r"D:\Jacky\Python\ADMM_P2P_Python\data"

primal = np.load(f"{datapath}/predictions/primal_pred.npy")
dual = np.load(f"{datapath}/predictions/dual_pred.npy")
decVar = np.load(f"{datapath}/training ready/decisionGRU.npy")
power_load = pd.read_csv(f"{DATA_DIR}/Power Consumption_33_bus.csv", header=0).values / 2
solar_sce = pd.read_csv(f"{DATA_DIR}/Solar_interpolated_6000.csv", header=0).values[SCE_START-1:SCE_END]

decVar = decVar[:,:,:,-1]
grid_trade = decVar[:,:96,:]
hour, num_user = power_load.shape

del decVar

In [5]:
# 1. Prepare Raw External Data (Before shuffling)
solar = solar_sce * SOLAR_SCALER
net_load = np.tile(power_load, (SCE_END-SCE_START+1, 1, 1))
pros_solar = int(np.ceil(num_user / 2)) - 1 # to match with Julia 1-inclusive indexing 
net_load[:, :, pros_solar:] -= solar[:, :, np.newaxis] # Shape: (1000, 48, 32)
net_load_wGrid = net_load.copy() # for later use in physics loss calculation

# 2. Calculate Grid Trade Variable (Sell - Buy) in Real Units
grid_trade = grid_trade[:,48:,:] - grid_trade[:,:48,:]

# 3. Include the final Grid Trade into Net Load (for training physics loss)
net_load_wGrid += grid_trade

net_load_wGrid.shape

(20, 48, 32)

In [6]:
data = load_scenario_data(DATA_DIR, BUS_SYS)

# === CES battery bounds ===
ub_CES = np.full((hour, num_user), BATTERY_CAP)
lb_CES = np.zeros((hour, num_user))

ub_CESc = np.full((hour, num_user), BATTERY_CAP / 3)  # charging bound
lb_CESc = np.zeros((hour, num_user))

ub_CESd = np.full((hour, num_user), BATTERY_CAP / 3)  # discharging bound
lb_CESd = np.zeros((hour, num_user))

# === Initialize other key parameters ===
beta_tnb = 1
P_CES0 = np.full(num_user, BATTERY_CAP / 2)
efficiency_CES = 1.0

# === Build prosumer parameters ===
param = {
    "ub_CES": ub_CES,
    "lb_CES": lb_CES,
    "ub_CESc": ub_CESc,
    "lb_CESc": lb_CESc,
    "ub_CESd": ub_CESd,
    "lb_CESd": lb_CESd,
    "hour": hour,
    "beta_tnb": beta_tnb,
    "CES0": P_CES0,
    "P2PTrade": P2P_TRADE,
    "efficiency_CES": efficiency_CES,
    "buy_priority": data["buy_priority"].T,
    "sell_priority": data["sell_priority"].T,
    "load_demamd": net_load.T,  # matches Julia's net_load'
}

# Feasibility Checks

In [8]:
feasibility_report = check_constraint_feasibility_np(primal, net_load_wGrid, tolerance=1e-3)
primal = post_process_constraint(primal, param, p2p_bal=True)
feasibility_report = check_constraint_feasibility_np(primal, net_load_wGrid, tolerance=1e-3, ver='Post-Processed')

--------------------------------------------------
PHYSICAL CONSTRAINT REPORT (Original)
--------------------------------------------------
1. Market Balance: ❌ FAIL
   Max Error: 0.230009 | Avg Error: 0.008630

2. Net Load Clearing: ❌ FAIL
   Max Error: 0.448328 | Avg Error: 0.008692
--------------------------------------------------
--------------------------------------------------
PHYSICAL CONSTRAINT REPORT (Post-Processed)
--------------------------------------------------
1. Market Balance: ✅ PASS
   Max Error: 0.000001 | Avg Error: 0.000000

2. Net Load Clearing: ❌ FAIL
   Max Error: 0.448328 | Avg Error: 0.009054
--------------------------------------------------


In [10]:
# save post-processed predictions
np.save(f"{datapath}/predictions/primal_pred_post.npy", primal)

# Performance Logging

In [4]:
# npz format
datapath = r"D:\Jacky\Data Output\ADMM_P2P\Database\LP_PrioGO_test_20_OldSame"
opt_iter = np.load(f"{datapath}/optimal_iter/conv_iter.npz")
obj_var = np.load(f"{datapath}/optimal_iter/obj_var.npz")
exe_time = np.load(f"{datapath}/optimal_iter/runtime.npz")

indices = [int(x-2) for x in opt_iter]
opt_obj_var = [obj_var[sce, i] for sce, i in enumerate(indices)]

print(opt_obj_var)
print(opt_iter)
print(exe_time)

[446.84243419190693, 447.96837292375903, 472.27052505293074, 461.10226388481595, 447.99407094566726, 458.1072173504615, 464.15185184988957, 449.7090082110244, 453.76361229174597, 456.7431993691055, 434.2872223255014, 451.5533841750289, 462.91777961774926, 451.4947215377261, 460.02020935668304, 461.681230302207, 450.1222497692016, 455.2895740499365, 463.1781507369618, 466.22223741778305]
[518. 518. 612. 843. 501. 518. 453. 622. 476. 648. 587. 450. 489. 491.
 756. 485. 609. 553. 477. 744.]
[195.359636  269.7976886 380.0653271 332.4674551 193.7409294 210.3351599
 189.8807108 310.8867891 195.9885554 273.4065458 243.5379294 186.0654944
 200.6914901 202.1245939 308.4255522 216.8526595 250.839642  226.8863333
 196.0033117 308.4755632]


In [38]:
# csv format
datapath = r"D:\Jacky\Data Output\ADMM_P2P\Database\Eval_PL_processed_beforeGO"
with open(f'{datapath}/config.json', 'r') as file:
    config = json.load(file)
start_sce_save, end_sce_save = config["sce_start"], config["sce_end"]
# gather_results(datapath, start_sce_save, end_sce_save)

opt_iter = pd.read_csv(f"{datapath}/optimal_iter/optimal_iter_{start_sce_save}to{end_sce_save}sce.csv").values.flatten()
obj_var = pd.read_csv(f"{datapath}/optimal_iter/objective_value_{start_sce_save}to{end_sce_save}sce.csv").values
exe_time = pd.read_csv(f"{datapath}/optimal_iter/execution_time_{start_sce_save}to{end_sce_save}sce.csv").values.flatten()

indices = [int(x-2) for x in opt_iter]
opt_obj_var = [obj_var[sce, i] for sce, i in enumerate(indices)]

if len(exe_time) != end_sce_save - start_sce_save + 1:
    print("Warning: Execution time length does not match number of scenarios. Please check the data.")
else:
    print(opt_obj_var)
    print(opt_iter)
    print(exe_time)

[446.80089899312736, 447.9372298793579, 472.26049284987977, 461.0829170086178, 447.9856038478042, 458.1084083848351, 464.14849583080576, 449.69722682866455, 453.68575643055, 456.7194375700237, 434.2580532532395, 451.5393402235314, 462.8754027666227, 451.4909203424373, 460.00728239210736, 461.66892244284617, 450.0307203100628, 455.306853962335, 463.1444862362828, 466.2244540171269]
[607 492 601 898 500 572 427 545 529 614 517 605 496 534 603 461 603 528
 490 659]
[141.5098318 112.5282132 139.8942395 207.7507054 116.9028449 133.2420491
  99.460736  128.727616  126.7865886 147.3954389 122.7617574 143.9041447
 117.8251881 129.3801881 146.717577  111.8207645 144.1617421 125.4077443
 117.9219205 160.1961244]
